## Exploring predictions

In [ ]:
import os
import sys
import gc
import random
import importlib as imp
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

import experiment_settings
import build_model
import train_model
import build_data

import rasterio
from rasterio.windows import Window
from rasterio.transform import Affine

from sklearn.metrics import mean_squared_error, mean_absolute_error

In [ ]:
print(f"python version = {sys.version}")
print(f"numpy version = {np.__version__}")
print(f"tensorflow version = {tf.__version__}")  

tf.config.set_visible_devices([], "GPU")  # turn-off tensorflow-metal if it is on
# print(tf.config.list_physical_devices('GPU'))

In [ ]:
# GET SETTINGS
EXP_NAME = "exp0"
settings = experiment_settings.get_settings(EXP_NAME)

SAVE_MODEL_DIRECTORY = "saved_models/"
DATA_DIRECTORY = "data/"
PREDICTIONS_DIRECTORY = "predictions/"
FIGURE_DIRECTORY = "figures/"

In [ ]:
# GET THE DATA
imp.reload(build_data)

for year in (2013,):#np.arange(2000, 2023):

    print(' --- ' + str(year) + '---')
    settings["testing_year"] = year
    settings["batch_size"] = 128

    # SET RANDOM SEEDS
    np.random.seed(settings["rng_seed"])
    random.seed(settings["rng_seed"])
    tf.random.set_seed(settings["rng_seed"])

    (tagyear_test, 
    taglat_test, 
    taglon_test,
    ) = build_data.make_sample_list(settings, evaluate_all=True)

    tfds_test = build_data.build_tf_dataset(settings, tagyear_test, taglat_test, taglon_test, settings["batch_size"], shuffle=False)
    tfds_test = tfds_test.prefetch(tf.data.AUTOTUNE)

    batch_shape = np.shape(next(tfds_test.as_numpy_iterator())[0])
    print(f"{batch_shape = }")

    # LOAD THE MODEL AND MAKE PREDICTIONS
    tf.keras.backend.clear_session()
    model = build_model.build_model(settings, input_shape=batch_shape[1:])

    checkpoint_dir = SAVE_MODEL_DIRECTORY + settings["exp_name"] + '/'
    model.load_weights(tf.train.latest_checkpoint(checkpoint_dir))

    predict_test = model.predict(tfds_test, verbose=1)[:,0]
    __ = gc.collect()

    # GET TIFF META DATA
    labels_year = settings["testing_year"]
    labels_filename = DATA_DIRECTORY + "hfi" + str(settings["testing_year"]) + "_merisINT.epsg4326.tif"
    if not os.path.isfile(labels_filename):
        labels_year = 2010
        labels_filename = DATA_DIRECTORY + "hfi" + str(2010) + "_merisINT.epsg4326.tif"

    with rasterio.open(labels_filename) as orig_tiff:

        ilat0, ilon0 = orig_tiff.index(settings["latlon_bounds"][2], settings["latlon_bounds"][0])
        ilat1, ilon1 = orig_tiff.index(settings["latlon_bounds"][3], settings["latlon_bounds"][1])
        lat0, lon0 = orig_tiff.xy(ilat0,ilon0)
        lat1, lon1 = orig_tiff.xy(ilat1,ilon1)    
        window = Window.from_slices((ilat0, ilat1 + 1), (ilon0, ilon1 + 1))
        orig_hfi_tile = orig_tiff.read(1, window=window)
        orig_hfi_tile = np.where(orig_hfi_tile < 255, orig_hfi_tile/50., 255)

    # SAVE THE TIFF
    width = ilon1 - ilon0 + 1
    height = ilat1 - ilat0 + 1
    res_lat = ((lat1-lat0)) / (height -1)
    res_lon = ((lon1-lon0)) / (width -1)

    predict_tile = np.reshape(predict_test, (height, width), order="C")
    predict_tile = np.where(orig_hfi_tile == 255, 255., predict_tile)

    predictions_filename = "predictions_JAKARTA_" + settings["exp_name"] + "_" + str(settings["testing_year"])
    meta_data = {}
    meta_data["nodata"] = 255.
    meta_data["width"] = width
    meta_data["height"] = height
    meta_data["driver"] = 'GTiff'
    meta_data["count"] = 1
    meta_data["crs"] = rasterio.CRS.from_epsg(4326)
    meta_data["dtype"] = predict_tile.dtype
    meta_data["transform"] = Affine.translation(lat0, lon0) * Affine.scale(res_lat, res_lon)

    with rasterio.open(PREDICTIONS_DIRECTORY + predictions_filename + ".tif","w",**meta_data) as dst:
        dst.write(predict_tile, 1)
        dst.set_band_description(1, 'mlHFI prediction')

    # PLOT THE RESULTS
    cmap = plt.get_cmap('PiYG_r')
    cmap.set_bad(color = 'lightgray', alpha = 1.)

    plt.figure(figsize=(10,5))

    plt.subplot(1,2,1)
    xplot = predict_tile
    # xplot[xplot==255] = np.nan
    plt.imshow(xplot, cmap=cmap, extent = [lon0, lon1, lat0, lat1])
    plt.title('mlHFI Predictions for ' + str(settings["testing_year"]))
    plt.clim(0,1)

    plt.subplot(1,2,2)
    xplot = orig_hfi_tile
    xplot[xplot==255] = np.nan
    plt.imshow(xplot, cmap=cmap, extent = [lon0, lon1, lat0, lat1])
    plt.title('HFI Labels for ' + str(labels_year))
    plt.clim(0,1)

    plt.savefig(FIGURE_DIRECTORY + predictions_filename + ".png")
    plt.show()
    # plt.close()